# Préparation des données d'entraînement

**Objectif** : fixer les lignes et les variables retenues pour les deux modèles décidés dans le
notebook 05.

| Service | Jeu de données | Cible |
|---|---|---|
| `/predict` | `data/agriculture-crop-yield/crop_yield.csv` | `Yield_tons_per_hectare` |
| `/recommend` | `data/processed/crop_yield_prediction_1990_2013.csv` | `yield_t_ha` |

Référentiel temporel de `/recommend` : le service produit ses recommandations en **2014** avec des
données disponibles jusqu'en **2013**. Une variable qui sert à prédire l'année *t* doit donc être
calculable avec des informations connues au plus tard en *t*−1, à l'entraînement comme à
l'inférence.

Les analyses ne sont pas refaites ici : elles viennent des notebooks 01 à 05. Aucun encodage,
aucune standardisation, aucune imputation statistique n'est appliquée — ces étapes seront
ajustées sur les seules données d'entraînement, plus tard.

## Imports

In [1]:
import json

import numpy as np
import pandas as pd

from agritech import geo
from agritech.config import AGRICULTURE_CROP_YIELD_FILENAME, PATHS, SEED

# 1. `/predict` — Agriculture CropYield

Régression du rendement d'une parcelle à partir de ses conditions et des pratiques déclarées.

## Lecture et contrôles de base

In [2]:
csv_path = PATHS.data_agriculture_crop_yield / AGRICULTURE_CROP_YIELD_FILENAME
df_agri = pd.read_csv(csv_path)

print("lignes x colonnes  :", df_agri.shape)
print("valeurs manquantes :", df_agri.isna().sum().sum())
print("doublons stricts   :", df_agri.duplicated().sum())
print("\ntypes :")
print(df_agri.dtypes.to_string())

lignes x colonnes  : (1000000, 10)
valeurs manquantes : 0
doublons stricts   : 0

types :
Region                     object
Soil_Type                  object
Crop                       object
Rainfall_mm               float64
Temperature_Celsius       float64
Fertilizer_Used              bool
Irrigation_Used              bool
Weather_Condition          object
Days_to_Harvest             int64
Yield_tons_per_hectare    float64


**Observations :**

- Aucun manquant, aucun doublon strict, types corrects : rien à corriger de ce côté.
- Les deux booléens sont déjà en `bool`, pas de conversion nécessaire à ce stade.

## Cible : les rendements négatifs

Le notebook 01 a repéré des rendements négatifs. Un rendement négatif n'a pas de sens physique,
contrairement à une valeur simplement atypique : les valeurs extrêmes détectées par l'IQR sont
conservées.

In [3]:
negatifs = df_agri[df_agri["Yield_tons_per_hectare"] < 0]

print(f"rendements négatifs : {len(negatifs)} ({len(negatifs) / len(df_agri):.3%})")
print(f"minimum de la cible : {df_agri['Yield_tons_per_hectare'].min():.2f} t/ha")
print(f"  avec engrais      : {negatifs['Fertilizer_Used'].sum()}")
print(f"  avec irrigation   : {negatifs['Irrigation_Used'].sum()}")
print(f"  moins de 200 mm   : {(negatifs['Rainfall_mm'] < 200).sum()}")

conserves = df_agri[df_agri["Yield_tons_per_hectare"] >= 0]
print(f"\nmoyenne de la cible : {df_agri['Yield_tons_per_hectare'].mean():.4f} "
      f"-> {conserves['Yield_tons_per_hectare'].mean():.4f}")
print(f"écart-type          : {df_agri['Yield_tons_per_hectare'].std():.4f} "
      f"-> {conserves['Yield_tons_per_hectare'].std():.4f}")

rendements négatifs : 231 (0.023%)
minimum de la cible : -1.15 t/ha
  avec engrais      : 0
  avec irrigation   : 0
  moins de 200 mm   : 223



moyenne de la cible : 4.6495 -> 4.6506
écart-type          : 1.6966 -> 1.6952


**Observations :**

- 231 lignes sur un million, toutes sans engrais ni irrigation et presque toutes sous 200 mm de
  pluie : ce sont les conditions les plus défavorables du jeu.
- Les retirer déplace la moyenne de la cible de 0,001 t/ha : l'effet est négligeable.
- **Décision : ces 231 lignes sont supprimées.** Les ramener à 0 supposerait de connaître le
  vrai rendement, ce qui n'est pas le cas. Aucune autre ligne n'est retirée, en particulier
  aucune sur un critère d'écart interquartile.

## Variables retenues

Le critère principal est la disponibilité au moment de la requête : une variable que
l'utilisateur ne peut pas renseigner ne peut pas entrer dans le modèle, quel que soit son
pouvoir explicatif.

| Variable | Disponible à la requête ? | Feature ? | Pourquoi |
|---|---|---|---|
| `Crop` | oui | garder | L'utilisateur choisit la culture : elle fait partie du contrat d'entrée. Aucun signal mesuré (notebooks 01 et 02), mais la conserver garde l'interface cohérente et rend ce constat vérifiable sur le modèle. |
| `Rainfall_mm` | oui | garder | Premier facteur du rendement (corrélation 0,765). |
| `Temperature_Celsius` | oui | garder | Effet faible mais régulier : +0,445 t/ha entre les tranches extrêmes. |
| `Fertilizer_Used` | oui | garder | +1,50 t/ha en moyenne. Pratique décidée par l'exploitant. |
| `Irrigation_Used` | oui | garder | +1,20 t/ha en moyenne. Idem. |
| `Soil_Type` | oui | garder | Propriété connue de la parcelle. Aucun signal mesuré, mais 6 modalités seulement et une entrée que l'utilisateur peut fournir. |
| `Region` | non | écarter | Macro-zone abstraite (North, East, South, West) sans correspondance géographique réelle : personne ne peut s'y situer. Aucun signal par ailleurs. |
| `Weather_Condition` | non | écarter | Météo dominante de la saison, connue seulement après coup, et indépendante de la pluie et de la température mesurées (notebook 01). |
| `Days_to_Harvest` | non | écarter | Durée constatée du cycle, connue à la récolte. Aucune relation avec le rendement, ni droite ni par tranches (notebook 01). |

In [4]:
FEATURES_PREDICT = [
    "Crop",
    "Soil_Type",
    "Rainfall_mm",
    "Temperature_Celsius",
    "Fertilizer_Used",
    "Irrigation_Used",
]
CIBLE_PREDICT = "Yield_tons_per_hectare"
ECARTEES_PREDICT = ["Region", "Weather_Condition", "Days_to_Harvest"]

df_predict = df_agri[df_agri[CIBLE_PREDICT] >= 0]

X_predict = df_predict[FEATURES_PREDICT]
y_predict = df_predict[CIBLE_PREDICT]

print(f"lignes           : {len(X_predict)} (sur {len(df_agri)}, soit {len(df_agri) - len(X_predict)} retirées)")
print(f"features         : {FEATURES_PREDICT}")
print(f"écartées         : {ECARTEES_PREDICT}")
print(f"cible            : {CIBLE_PREDICT}")
print(f"manquants dans X : {X_predict.isna().sum().sum()}")
X_predict.head()

lignes           : 999769 (sur 1000000, soit 231 retirées)
features         : ['Crop', 'Soil_Type', 'Rainfall_mm', 'Temperature_Celsius', 'Fertilizer_Used', 'Irrigation_Used']
écartées         : ['Region', 'Weather_Condition', 'Days_to_Harvest']
cible            : Yield_tons_per_hectare
manquants dans X : 0


,Crop,Soil_Type,Rainfall_mm,Temperature_Celsius,Fertilizer_Used,Irrigation_Used
0,Cotton,Sandy,897.077239,27.676966,False,True
1,Rice,Clay,992.673282,18.026142,True,True
2,Barley,Loam,147.998025,29.794042,False,False
3,Soybean,Sandy,986.866331,16.644190,False,True
4,Wheat,Silt,730.379174,31.620687,True,True


**Observations :**

- 999 769 lignes, 6 features, aucune valeur manquante.
- Deux variables catégorielles (`Crop`, `Soil_Type`) et deux booléennes restent brutes :
  l'encodage sera ajusté sur le train seul.

# 2. `/recommend` — historique consolidé

Service à l'échelle du **pays** : l'utilisateur fournit son pays, l'application y associe le
contexte historique (température, pluie, pesticides), le modèle prédit un rendement pour chaque
culture candidate et les rendements sont triés. `iso3` est la clé qui permet de retrouver ce
contexte ; ce n'est pas une variable du modèle. La source couvre 1990-2013 (notebook 03) ; le périmètre
d'entraînement commencera en 1991, après création des variables historiques.

## Lecture et contrôles de base

In [5]:
DEBUT, FIN = 1990, 2013
df_hist = pd.read_csv(PATHS.data_processed / f"crop_yield_prediction_{DEBUT}_{FIN}.csv")

print("lignes x colonnes     :", df_hist.shape)
print("doublons sur la clé   :", df_hist.duplicated(["iso3", "year", "crop"]).sum())
print("cultures / pays       :", df_hist.crop.nunique(), "/", df_hist.iso3.nunique())
print("rendements négatifs   :", (df_hist.yield_t_ha < 0).sum())
print("rendements nuls       :", (df_hist.yield_t_ha == 0).sum())
print("\nvaleurs manquantes (%) :")
print((df_hist.isna().mean() * 100).round(1).to_string())

lignes x colonnes     : (22679, 8)
doublons sur la clé   : 0
cultures / pays       : 10 / 168
rendements négatifs   : 0
rendements nuls       : 8

valeurs manquantes (%) :
iso3             0.0
area             0.0
year             0.0
crop             0.0
yield_t_ha       0.0
avg_temp        19.5
rain_mm          5.0
pesticides_t    13.1


**Observations :**

- Clé `iso3 + year + crop` unique, aucun rendement négatif : rien d'équivalent au problème du
  premier jeu.
- Huit rendements nuls, une récolte déclarée nulle restant possible. Ils disparaîtront avec le
  nettoyage des manquants, leurs pays n'ayant ni température ni pluie.
- Trois variables incomplètes : `avg_temp` 19,5 %, `pesticides_t` 13,1 %, `rain_mm` 5,0 %.

## Nature des valeurs manquantes

Avant de décider quoi en faire, il faut savoir si les manquants sont des trous dans une série ou
une absence de couverture du pays : le second cas ne se complète pas sans source externe.

In [6]:
def couverture(colonne):
    """Répartit les manquants d'une colonne entre pays non couverts et trous partiels."""
    par_pays = df_hist.groupby("iso3")[colonne].agg(["count", "size"])
    aucun = par_pays[par_pays["count"] == 0]
    partiel = par_pays[(par_pays["count"] > 0) & (par_pays["count"] < par_pays["size"])]
    return pd.Series({
        "lignes manquantes": df_hist[colonne].isna().sum(),
        "pays sans aucune valeur": len(aucun),
        "lignes de ces pays": aucun["size"].sum(),
        "pays partiellement couverts": len(partiel),
    })


pd.DataFrame({colonne: couverture(colonne) for colonne in ["avg_temp", "rain_mm", "pesticides_t"]})

,avg_temp,rain_mm,pesticides_t
lignes manquantes,4415,1130,2975
pays sans aucune valeur,36,1,24
lignes de ces pays,4415,177,2975
pays partiellement couverts,0,163,0


In [7]:
# rain_mm est la seule colonne à trous partiels : d'où viennent-ils ?
sans_pluie = df_hist[df_hist["rain_mm"].isna()]
print("années les plus touchées :")
print(sans_pluie["year"].value_counts().head(3).to_string())

pays_sans_pluie = df_hist.groupby("area")["rain_mm"].count()
print("\npays sans aucune valeur de pluie :", pays_sans_pluie[pays_sans_pluie == 0].index.tolist())
print("valeurs distinctes de rain_mm par pays :", df_hist.groupby("iso3")["rain_mm"].nunique().max(), "au maximum")

années les plus touchées :
year
2003    954
1990     11
1991     10

pays sans aucune valeur de pluie : ['New Caledonia']
valeurs distinctes de rain_mm par pays : 1 au maximum


**Observations :**

- `avg_temp` et `pesticides_t` manquent **par pays entier** : 36 pays sans aucune température,
  24 sans aucun tonnage. Aucun trou partiel. Il n'y a donc rien à compléter à partir des données
  elles-mêmes.
- `rain_mm` est différent : l'année **2003 est absente de `rainfall.csv`**, donc de toutes les
  lignes 2003. Le reste vient d'un seul pays sans valeur, la Nouvelle-Calédonie.
- `rain_mm` ne prend **qu'une seule valeur par pays** : c'est la normale climatique déjà
  identifiée au notebook 03.

### Reprise de l'année 2003

`rain_mm` étant constante par pays, recopier la valeur du pays comble 2003 sans rien estimer.
Ce n'est pas une imputation statistique : aucune moyenne n'est calculée, la valeur existe déjà
dans la colonne.

In [8]:
avant = df_hist["rain_mm"].isna().sum()
df_hist["rain_mm"] = df_hist.groupby("iso3")["rain_mm"].transform("first")

print(f"rain_mm manquante : {avant} -> {df_hist['rain_mm'].isna().sum()}")
print("années couvertes  :", df_hist.dropna(subset=["rain_mm"])["year"].nunique())

rain_mm manquante : 1130 -> 177
années couvertes  : 24


## Pays couverts

`avg_temp` et `pesticides_t` manquent par pays entier : rien ne permet de les compléter depuis les
données. Le périmètre de `/recommend` est donc celui des pays qui ont les trois variables de
contexte.

In [9]:
avec_climat = df_hist.dropna(subset=["avg_temp", "rain_mm"])
avec_contexte = df_hist.dropna(subset=["avg_temp", "rain_mm", "pesticides_t"])

print(f"température + pluie              : {len(avec_climat)} lignes, {avec_climat.iso3.nunique()} pays")
print(f"température + pluie + pesticides : {len(avec_contexte)} lignes, {avec_contexte.iso3.nunique()} pays")

sans_pesticides = sorted(set(avec_climat["area"]) - set(avec_contexte["area"]))
print(f"\npays avec climat mais sans pesticides ({len(sans_pesticides)}) :")
print(sans_pesticides)

température + pluie              : 18264 lignes, 132 pays
température + pluie + pesticides : 16357 lignes, 117 pays

pays avec climat mais sans pesticides (15) :
['Afghanistan', 'Bosnia and Herzegovina', 'Democratic Republic of the Congo', 'Equatorial Guinea', 'Gabon', 'Georgia', 'Liberia', 'Mongolia', 'Nigeria', 'Philippines', 'Serbia', 'Sierra Leone', 'Somalia', 'United Arab Emirates', 'Uzbekistan']


**Observations :**

- Exiger les pesticides coûte 15 pays et 1 907 lignes. Ces 15 pays sont examinés en fin de
  notebook ; aucune estimation n'est faite pour eux.
- Périmètre retenu : 117 pays. Il perdra encore une année avec les features historiques.

## Features historiques

Règle : pour prédire l'année *t*, seules les années ≤ *t*−1 sont utilisables. Les deux variables
annuelles du contexte sont donc remplacées par la **moyenne des trois années précédentes au
plus** (au moins une), calculée par pays avec `shift(1)` puis `rolling(3)`. Pour 2013, la fenêtre
est 2010-2012 ; pour une requête en 2014, elle sera 2011-2013.

- `temp_hist` : moyenne de `avg_temp` sur cette fenêtre. La température N−1 donnerait la même
  chose (corrélation 0,999, écart médian 0,19 °C) : une seule recette pour les deux variables.
- `pest_hist` : moyenne de `pesticides_t` sur cette fenêtre, puis `log_pest_hist = log1p(pest_hist)`.
  La distribution des tonnages est très asymétrique et le logarithme réduit le poids des très
  grandes valeurs.
- `rain_mm` est une constante par pays : aucun décalage nécessaire.

In [10]:
def moyenne_historique(serie):
    """Moyenne des 3 années précédentes au plus, sans l'année courante."""
    return serie.shift(1).rolling(3, min_periods=1).mean()


contexte_pays = (
    df_hist[["iso3", "year", "avg_temp", "pesticides_t"]]
    .drop_duplicates(["iso3", "year"])
    .sort_values(["iso3", "year"])
)
contexte_pays["temp_hist"] = contexte_pays.groupby("iso3")["avg_temp"].transform(moyenne_historique)
contexte_pays["pest_hist"] = contexte_pays.groupby("iso3")["pesticides_t"].transform(moyenne_historique)
contexte_pays["log_pest_hist"] = np.log1p(contexte_pays["pest_hist"])

df_hist = df_hist.merge(
    contexte_pays[["iso3", "year", "temp_hist", "pest_hist", "log_pest_hist"]],
    on=["iso3", "year"],
    how="left",
)
print("lignes après jointure :", len(df_hist))
print("colonnes              :", df_hist.columns.tolist())

lignes après jointure : 22679
colonnes              : ['iso3', 'area', 'year', 'crop', 'yield_t_ha', 'avg_temp', 'rain_mm', 'pesticides_t', 'temp_hist', 'pest_hist', 'log_pest_hist']


In [11]:
exemple = contexte_pays[contexte_pays["iso3"] == "FRA"]
annees = [1990, 1991, 1992, 1993, 2010, 2011, 2012, 2013]
colonnes = ["year", "avg_temp", "temp_hist", "pesticides_t", "pest_hist", "log_pest_hist"]

print("France :")
print(exemple[exemple["year"].isin(annees)][colonnes].round(2).to_string(index=False))

France :
 year  avg_temp  temp_hist  pesticides_t  pest_hist  log_pest_hist
 1990     11.96        NaN      97701.00        NaN            NaN
 1991     10.60      11.96     103434.00   97701.00          11.49
 1992     11.15      11.28      85249.00  100567.50          11.52
 1993     10.70      11.24      91953.00   95461.33          11.47
 2010     10.41      11.50      61903.00   73169.33          11.20
 2011     12.33      11.05      61039.00   68052.00          11.13
 2012     11.22      11.40      63547.59   62206.00          11.04
 2013     11.01      11.32      66497.29   62163.20          11.04


### Contrôles anti-fuite

Trois vérifications simples : les années sont bien consécutives par pays (sinon `shift(1)` ne
désignerait pas *t*−1), la première année de chaque pays n'a pas d'historique, et la feature se
retrouve en recalculant à la main la moyenne de *t*−3 à *t*−1.

In [12]:
consecutives = contexte_pays.groupby("iso3")["year"].diff().dropna().eq(1).all()
print("années consécutives par pays            :", consecutives)

premieres = contexte_pays.groupby("iso3").head(1)
print("features vides sur la première année    :",
      premieres["temp_hist"].isna().all() and premieres["pest_hist"].isna().all())

ecart_max = 0
for _, ligne in contexte_pays.dropna(subset=["pest_hist"]).sample(300, random_state=SEED).iterrows():
    fenetre = contexte_pays[
        (contexte_pays["iso3"] == ligne["iso3"])
        & contexte_pays["year"].between(ligne["year"] - 3, ligne["year"] - 1)
    ]
    ecart_max = max(ecart_max, abs(fenetre["pesticides_t"].mean() - ligne["pest_hist"]))
print("écart max avec la moyenne t-3..t-1 recalculée :", round(ecart_max, 6))

années consécutives par pays            : True
features vides sur la première année    : True
écart max avec la moyenne t-3..t-1 recalculée : 0.0


**Observations :**

- La feature d'une année *t* est bien la moyenne de *t*−3 à *t*−1 : écart nul au recalcul manuel,
  aucune valeur de l'année *t* n'y entre.
- Pour la France en 2013, `pest_hist` est la moyenne de 2010, 2011 et 2012 ; en 2014, ce serait
  2011-2013.
- 1990 n'a aucune année antérieure dans le jeu : ses features sont vides et l'année sortira du
  périmètre.

## Périmètre d'entraînement

Lignes complètes sur les trois variables de contexte historiques.

In [13]:
df_recommend = df_hist.dropna(subset=["temp_hist", "rain_mm", "log_pest_hist"])

print(f"lignes    : {len(df_recommend)} sur {len(df_hist)} ({len(df_recommend) / len(df_hist):.1%})")
print(f"pays      : {df_recommend.iso3.nunique()}")
print(f"années    : {df_recommend.year.min()} - {df_recommend.year.max()} ({df_recommend.year.nunique()} distinctes)")
print(f"cultures  : {df_recommend.crop.nunique()}")
print(f"rendements <= 0 restants : {(df_recommend.yield_t_ha <= 0).sum()}")

lignes    : 15664 sur 22679 (69.1%)
pays      : 117
années    : 1991 - 2013 (23 distinctes)
cultures  : 10
rendements <= 0 restants : 0


**Observations :**

- 15 664 lignes, 117 pays, 10 cultures, 1991-2013 : 1990 disparaît faute d'année antérieure.
- 2013 est conservée dans le périmètre final et pourra être réservée au test temporel.

## Cultures candidates

Les dix cultures de l'historique restent candidates. Reste à vérifier qu'aucune n'est trop mal
couverte pour donner une prédiction défendable.

In [14]:
couverture_cultures = df_recommend.groupby("crop").agg(
    lignes=("yield_t_ha", "size"),
    pays=("iso3", "nunique"),
    années=("year", "nunique"),
    rendement_médian=("yield_t_ha", "median"),
).sort_values("lignes", ascending=False)

couverture_cultures.round(2)

,lignes,pays,années,rendement_médian
crop,,,,
Maize,2407,109,23,2.60
Potatoes,2401,109,23,15.95
Wheat,2095,95,23,2.48
"Rice, paddy",1815,82,23,3.49
Sorghum,1713,80,23,1.27
Soybeans,1589,73,23,1.60
Sweet potatoes,1368,61,23,8.25
Cassava,1127,49,23,10.00
Plantains and others,602,27,23,8.40


**Observations :**

- Les dix cultures couvrent les 23 années. Aucune n'est écartée.
- Igname (547 lignes, 25 pays) et plantain (602 lignes, 27 pays) restent les plus minces : leurs
  prédictions seront à surveiller.
- Les rendements médians vont de 1,3 t/ha (sorgho) à 16 t/ha (pomme de terre) : en tonnes par
  hectare, les tubercules domineront presque toujours le classement. Rien n'est corrigé ici —
  rendement n'est pas rentabilité, et aucune donnée économique n'est disponible.

## Domaine climatique des cultures

Le service évaluera les dix cultures pour chaque pays, y compris là où une culture n'a jamais été
observée. Le domaine historique de chaque culture — quantiles 5 % et 95 % de la température et de
la pluie — prépare la future règle : une culture évaluée loin de son domaine devra être signalée
comme extrapolée, ou exclue du classement. Le seuil n'est pas fixé ici.

In [15]:
domaine = df_recommend.groupby("crop").agg(
    temp_p5=("temp_hist", lambda s: s.quantile(0.05)),
    temp_p95=("temp_hist", lambda s: s.quantile(0.95)),
    pluie_p5=("rain_mm", lambda s: s.quantile(0.05)),
    pluie_p95=("rain_mm", lambda s: s.quantile(0.95)),
)

domaine.sort_values("temp_p5").round(0)

,temp_p5,temp_p95,pluie_p5,pluie_p95
crop,,,,
Wheat,6.0,27.0,89.0,1996.0
Potatoes,6.0,28.0,92.0,2274.0
Maize,8.0,28.0,92.0,2387.0
Soybeans,8.0,27.0,250.0,2280.0
Sorghum,9.0,28.0,92.0,2280.0
"Rice, paddy",9.0,28.0,151.0,2702.0
Sweet potatoes,11.0,28.0,197.0,2689.0
Yams,16.0,28.0,282.0,3142.0
Plantains and others,17.0,28.0,1071.0,2387.0


**Observations :**

- Blé, pomme de terre et maïs sont observés dès 6-8 °C ; igname, plantain et manioc ne le sont
  jamais sous 16-17 °C, et le plantain jamais sous 1 000 mm de pluie.
- Le haut de la plage est commun à toutes les cultures (27-28 °C) : la limite utile est surtout
  le froid et la sécheresse.
- Un pays froid recevrait quand même une prédiction pour l'igname : sans garde-fou, elle ne
  reposerait sur aucune observation.

## Variables retenues

Le critère reste la disponibilité au moment de la requête, mais l'utilisateur ne saisit que son
pays : le reste du contexte est retrouvé par l'application dans l'historique, avec des valeurs
connues au plus tard en *t*−1.

| Variable | Disponible à la requête ? | Feature ? | Pourquoi |
|---|---|---|---|
| `crop` | oui, générée par le service | garder | Une ligne par culture candidate dans le même contexte : c'est la variable qui porte le classement. |
| `temp_hist` | oui, retrouvée par le pays | garder | Température des trois dernières années connues. |
| `rain_mm` | oui, retrouvée par le pays | garder | Normale climatique du pays, constante dans le temps. |
| `log_pest_hist` | oui, retrouvée par le pays | garder | Niveau de pesticides des trois dernières années connues, en logarithme. Corrélation la plus élevée parmi les variables de contexte testées dans le notebook 05 ; elle traduit un niveau national d'intensification, confondu avec la taille du pays. |
| `avg_temp`, `pesticides_t` | non | écarter | Valeurs de l'année *t*, connues seulement après coup : remplacées par leurs versions historiques. |
| `year` | oui | à tester | Le référentiel 2014 ne demande qu'une extrapolation d'un an, et les rendements progressent sur la période (notebook 03). Conservée dans le contexte ; son entrée dans le modèle sera décidée en comparant les modèles. |
| `iso3`, `area` | oui | non | Clé technique pour retrouver le contexte du pays, et identifiants pour la validation. Comme feature, le modèle deviendrait une table de correspondance sans généralisation. |

In [16]:
FEATURES_RECOMMEND = ["crop", "temp_hist", "rain_mm", "log_pest_hist"]
CIBLE_RECOMMEND = "yield_t_ha"
CONTEXTE_RECOMMEND = ["iso3", "area", "year"]   # year : feature candidate, à trancher en modélisation

X_recommend = df_recommend[FEATURES_RECOMMEND]
y_recommend = df_recommend[CIBLE_RECOMMEND]
contexte_recommend = df_recommend[CONTEXTE_RECOMMEND]

print(f"lignes           : {len(X_recommend)}")
print(f"features         : {FEATURES_RECOMMEND}")
print(f"écartées         : ['avg_temp', 'pesticides_t'] (remplacées par leurs versions historiques)")
print(f"contexte gardé   : {CONTEXTE_RECOMMEND}")
print(f"cible            : {CIBLE_RECOMMEND}")
print(f"manquants dans X : {X_recommend.isna().sum().sum()}")
X_recommend.head()

lignes           : 15664
features         : ['crop', 'temp_hist', 'rain_mm', 'log_pest_hist']
écartées         : ['avg_temp', 'pesticides_t'] (remplacées par leurs versions historiques)
contexte gardé   : ['iso3', 'area', 'year']
cible            : yield_t_ha
manquants dans X : 0


,crop,temp_hist,rain_mm,log_pest_hist
97,Maize,16.370000,1485.0,4.804021
98,Maize,15.865000,1485.0,4.804021
99,Maize,15.930000,1485.0,4.804021
100,Maize,15.823333,1485.0,4.804021
101,Maize,16.356667,1485.0,5.001707


**Observations :**

- 15 664 lignes, 4 features, aucune valeur manquante.
- Le contexte reste à côté du modèle : `iso3` pour retrouver les features d'un pays et grouper la
  validation, `year` pour le découpage temporel et comme feature candidate.

## Pays sans pesticides : une expérience

Quinze pays ont un climat mais aucun tonnage de pesticides. Avant de les déclarer hors service, on
regarde si leurs voisins permettraient une estimation défendable. Deux voisinages sont comparés
sur trois pays de climats différents : les pays les plus proches en température et pluie (distance
euclidienne sur les deux variables standardisées), et les pays frontaliers d'après les contours
Natural Earth déjà utilisés pour les cartes. **Aucune valeur n'est imputée** : c'est une
exploration.

In [17]:
pays = (
    df_hist.groupby(["iso3", "area"])
    .agg(temp=("avg_temp", "mean"), pluie=("rain_mm", "first"), pesticides=("pesticides_t", "mean"))
    .reset_index()
)
avec_pesticides = pays.dropna(subset=["temp", "pluie", "pesticides"])

# voisins de contexte : distance sur température et pluie standardisées
moyenne = avec_pesticides[["temp", "pluie"]].mean()
ecart_type = avec_pesticides[["temp", "pluie"]].std()
z_avec = (avec_pesticides[["temp", "pluie"]] - moyenne) / ecart_type
climat = pays.set_index("iso3")[["temp", "pluie"]]


def voisins_de_contexte(code, k=3):
    """Les k pays couverts les plus proches en température et pluie."""
    z = (climat.loc[code] - moyenne) / ecart_type
    distance = ((z_avec - z) ** 2).sum(axis=1) ** 0.5
    return avec_pesticides.assign(distance=distance).nsmallest(k, "distance")


# voisins géographiques : pays partageant des sommets de frontière dans le GeoJSON
geojson = json.loads(geo.GEOJSON_PAR_DEFAUT.read_text(encoding="utf-8"))
sommets = {}
for f in geojson["features"]:
    proprietes = f["properties"]
    code = proprietes["ISO_A3_EH"] if proprietes["ISO_A3_EH"] != "-99" else proprietes["ADM0_A3"]
    geometrie = f["geometry"]
    polygones = geometrie["coordinates"] if geometrie["type"] == "MultiPolygon" else [geometrie["coordinates"]]
    sommets[code] = {tuple(point) for polygone in polygones for anneau in polygone for point in anneau}


def frontaliers(code):
    """Pays partageant au moins deux sommets de frontière avec `code`."""
    return [autre for autre, s in sommets.items() if autre != code and len(sommets[code] & s) >= 2]

In [18]:
colonnes = ["area", "temp", "pluie", "pesticides"]

for code in ["BIH", "AFG", "COD"]:
    cible = pays.set_index("iso3").loc[code]
    print(f"\n=== {cible['area']} — température {cible['temp']:.1f} °C, pluie {cible['pluie']:.0f} mm, pesticides : aucun")
    print("voisins de contexte :")
    print(voisins_de_contexte(code)[colonnes].round(1).to_string(index=False))
    print("frontaliers :")
    print(pays[pays["iso3"].isin(frontaliers(code))][colonnes].round(1).to_string(index=False))


=== Bosnia and Herzegovina — température 10.1 °C, pluie 1028 mm, pesticides : aucun
voisins de contexte :
   area  temp  pluie  pesticides
Croatia  10.7 1113.0      2401.0
Ireland   9.5 1118.0      2432.4
Austria   9.1 1110.0      3579.7
frontaliers :
      area  temp  pluie  pesticides
   Croatia  10.7 1113.0      2401.0
Montenegro  11.5  241.0        37.0
    Serbia  11.3  686.0         NaN

=== Afghanistan — température 15.3 °C, pluie 327 mm, pesticides : aucun
voisins de contexte :
                      area  temp  pluie  pesticides
Iran (Islamic Republic of)  15.0  228.0      6990.2
                 Australia  16.6  534.0     33086.1
                    Turkey  15.1  593.0     33300.2
frontaliers :
                      area  temp  pluie  pesticides
           China, mainland  13.2  645.0   1327576.5
Iran (Islamic Republic of)  15.0  228.0      6990.2
                  Pakistan  24.7  494.0      8682.8
                Tajikistan   8.5  691.0       564.6
              Turkmenistan

**Observations :**

- Bosnie-Herzégovine : voisins de contexte (Croatie, Irlande, Autriche) entre 2 400 et 3 600 t,
  cohérents ; mais ses frontaliers vont de 37 t (Monténégro) à 2 400 t (Croatie), et la Serbie n'a
  pas de valeur.
- Afghanistan : voisins de contexte de 7 000 à 33 000 t ; frontaliers de 565 t (Tadjikistan) à
  1,3 million (Chine).
- RD Congo : ses voisins de contexte sont le Congo (11 t), le Brésil (190 000 t) et le Cameroun
  (760 t). Même climat, niveaux sans rapport.
- Le tonnage national dépend d'abord de la taille et de l'économie du pays, pas de son climat : ni
  le voisinage climatique ni le voisinage géographique ne donnent un niveau prévisible. Une
  moyenne de voisins serait arbitraire.
- **Décision : aucune imputation.** `/recommend` couvre 117 pays ; les 15 autres devront recevoir
  une réponse explicite d'indisponibilité.

# 3. Ce qui reste à faire au moment de la modélisation

Rien n'a été appris sur les données ici : suppression de lignes, sélection de colonnes, recopie
d'une constante par pays pour `rain_mm`, et moyennes glissantes qui ne regardent que le passé de
chaque ligne.

Côté `/predict` : encodage de `Crop` et `Soil_Type`, mise à l'échelle si le modèle en a besoin,
découpage train / test — le tout ajusté sur les seules données d'entraînement.

Côté `/recommend`, la validation devra au minimum :

- découper dans le temps : entraînement sur le passé, validation sur des années antérieures à
  2013, **2013 réservée au test final**. Un découpage aléatoire mélangerait des lignes quasi
  identiques d'un même pays d'une année sur l'autre ;
- comparer à deux références simples : la moyenne par culture, et si possible la persistance —
  le dernier rendement connu du pays pour la même culture ;
- mesurer la régression (RMSE, MAE) et le classement (top-k, corrélation de rang) ;
- tester `year` comme feature ;
- en contrôle complémentaire, une validation groupée par pays pour mesurer la généralisation à
  un pays inconnu — utile, mais secondaire par rapport au découpage temporel.

Restent aussi à définir le seuil du garde-fou de domaine climatique, et la table de contexte par
pays figée à fin 2013 que l'API utilisera.

# 4. Faut-il enregistrer de nouveaux fichiers ?

Non, à ce stade.

- `/predict` part du fichier source et ne retire que 231 lignes.
- `/recommend` part de `crop_yield_prediction_1990_2013.csv`, déjà produit par le notebook 04 ;
  les features historiques se recalculent en quelques lignes.
- Un fichier intermédiaire créerait surtout un risque de désynchronisation avec les sources.

Si les notebooks de modélisation répètent ce code, il ira dans `src/agritech/`. La table de
contexte par pays pour l'API sera, elle, un artefact à produire au moment du déploiement.

# Conclusion

In [19]:
print("/predict   — Agriculture CropYield")
print(f"  lignes   : {len(X_predict)} ({len(df_agri) - len(X_predict)} rendements négatifs retirés)")
print(f"  cible    : {CIBLE_PREDICT}")
print(f"  features : {FEATURES_PREDICT}")
print(f"  écartées : {ECARTEES_PREDICT}")

print("\n/recommend — historique consolidé, inférence 2014 sur données <= 2013")
print(f"  lignes   : {len(X_recommend)} ({df_recommend.crop.nunique()} cultures, {df_recommend.iso3.nunique()} pays, "
      f"{df_recommend.year.min()}-{df_recommend.year.max()})")
print(f"  cible    : {CIBLE_RECOMMEND}")
print(f"  features : {FEATURES_RECOMMEND}")
print(f"  contexte : {CONTEXTE_RECOMMEND} (year : feature candidate)")

/predict   — Agriculture CropYield
  lignes   : 999769 (231 rendements négatifs retirés)
  cible    : Yield_tons_per_hectare
  features : ['Crop', 'Soil_Type', 'Rainfall_mm', 'Temperature_Celsius', 'Fertilizer_Used', 'Irrigation_Used']
  écartées : ['Region', 'Weather_Condition', 'Days_to_Harvest']

/recommend — historique consolidé, inférence 2014 sur données <= 2013
  lignes   : 15664 (10 cultures, 117 pays, 1991-2013)
  cible    : yield_t_ha
  features : ['crop', 'temp_hist', 'rain_mm', 'log_pest_hist']
  contexte : ['iso3', 'area', 'year'] (year : feature candidate)


**Décisions de préparation :**

- `/predict` : 231 rendements négatifs supprimés, aucune suppression sur critère d'écart
  interquartile. `Region`, `Weather_Condition` et `Days_to_Harvest` écartées faute d'être
  renseignables à la requête.
- `/recommend` : service à l'échelle du pays, référentiel « inférence 2014, données ≤ 2013 ».
  Température et pesticides remplacés par la moyenne des trois années précédentes au plus,
  pesticides en logarithme ; `rain_mm` reprise pour 2003. Pays sans température ou sans
  pesticides hors périmètre : 117 pays, aucune imputation.
- Les dix cultures restent candidates ; le domaine climatique de chacune est documenté pour un
  futur garde-fou.
- `iso3` est une clé, pas une feature ; `year` reste une feature candidate à trancher en
  modélisation.
- Aucun encodage ni statistique apprise : tout le prétraitement appris reste à faire sur les
  données d'entraînement.